# 06: V18 figures from the Kwant pilot (notebook 05)

Regenerates every Kwant figure used in RHG V18 from the pilot output of `05_kwant_RHG_thermal_metal.ipynb`. **No new transport calculation is run**: all transmissions, winding numbers, Bott indices and KPM spectra are read from `out_05_rhg_pilot/` (CSV files and the task cache). The only Kwant evaluation is the small current map of Fig. 5b (one 64 x 42 cylinder, seconds).

**Run from the folder that contains `rhg_kwant_core.py` and `out_05_rhg_pilot/`** (the folder used for notebook 05). Optional environment variables: `RHG_PILOT_DIR`, `RHG_CORE_DIR`, `RHG_FIG_DIR` (default `plots_RHG`).

Outputs (in `plots_RHG/`):

| file | use in V18 |
|---|---|
| `fig_kwant_disorder_window.pdf` | main Fig. 4: disorder test of the transport window |
| `fig_kwant_writable_walls.pdf` | main Fig. 5: written walls and domain networks |
| `fig_kwant_model_bridge.pdf` | appendix: lattice-resolved model, channel counting, real-space Hall function |
| `fig_kwant_scaling_appendix.pdf` | appendix: full scaling data and class-D calibration |
| `kwant_v18_numbers.json` | every number quoted in the V18 text |

Colour roles: blue = gapped point G (and its disorder ramp), one-hue orange ramp = pocket size P1 to P4, aqua = third role.

In [1]:
import os, sys, json, hashlib, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import ndimage

warnings.filterwarnings("ignore")
PILOT = Path(os.environ.get("RHG_PILOT_DIR", "out_05_rhg_pilot"))     # output folder of notebook 05 (pilot)
CORE_DIR = Path(os.environ.get("RHG_CORE_DIR", "."))                   # folder containing rhg_kwant_core.py
FIGDIR = Path(os.environ.get("RHG_FIG_DIR", "plots_RHG")); FIGDIR.mkdir(exist_ok=True)
CACHE = PILOT / "cache"
sys.path.insert(0, str(CORE_DIR.resolve()))
import rhg_kwant_core as rc
import kwant

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm", "font.size": 9.5, "axes.labelsize": 10.5, "axes.titlesize": 10,
    "legend.fontsize": 7.8, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "axes.linewidth": 0.9, "lines.linewidth": 1.6, "lines.markersize": 4.2,
    "figure.dpi": 130, "savefig.dpi": 400, "savefig.bbox": "tight", "legend.frameon": False,
})
# colour roles (validated): gapped point = blue; pocket points = one-hue orange ordinal ramp (pocket size);
# third role = aqua. Disorder strength at G = one-hue blue ordinal ramp.
C_G = "#2a78d6"
C_P = {"P1": "#f0a070", "P2": "#eb6834", "P3": "#bf4515", "P4": "#7f2a0a"}
C_AQ = "#1baf7a"
BLUE_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
MK = {"G": "o", "P1": "s", "P2": "D", "P3": "^", "P4": "v"}
INK2 = "#52514e"

# ---- identical definitions to notebook 05 (cache keys must match)
LR = dict(r_in=0.45, r_out=1.35, q=0.90, D0=1.0, chi=-1.0); LR["Qx"] = 0.1 * LR["r_out"]
POINTS = {"G": 1.0, "P1": 2.2, "P2": 2.5, "P3": 3.0, "P4": 4.0}
E0, BUFFER = 1e-4, 2
wy = lambda L: max(4, int(round(L / np.sqrt(3))))


def _hash(name, kw):
    return hashlib.sha1(json.dumps([name, kw], sort_keys=True, default=float).encode()).hexdigest()[:20]


def load(name, kw):
    f = CACHE / f"{name}_{_hash(name, kw)}.pkl"
    return pickle.loads(f.read_bytes()) if f.exists() else None


def chunks(seq, n):
    seq = list(seq); return [seq[i:i + n] for i in range(0, len(seq), n)]


def seeds_for(U, n):
    return [0] if U == 0 else list(range(1, n + 1))


def cyl_key(w, L, leads="metal", **extra):
    g = dict(LR, w=float(w), kind="cylinder", Lx=int(L), Wy=wy(L), frame="lab", buffer=BUFFER, lead_type=leads)
    g.update(extra); return rc.gkey(g)


def torus_key(w, L):
    return rc.gkey(dict(LR, w=float(w), kind="torus", Lx=int(L), Wy=wy(L), frame="gauge"))


def panel(ax, s):
    ax.text(-0.17, 1.03, s, transform=ax.transAxes, fontsize=11, fontweight="bold", va="bottom")


g = pd.read_csv(PILOT / "main_g.csv"); g["lg"] = np.log(g.g.clip(lower=1e-300))
q = pd.read_csv(PILOT / "main_Q.csv")
bott = pd.read_csv(PILOT / "main_bott.csv")
dos0 = pd.read_csv(PILOT / "main_dos0.csv")
walls = pd.read_csv(PILOT / "walls.csv")
tex = pd.read_csv(PILOT / "textures_g.csv")
cal = pd.read_csv(PILOT / "calibration_g.csv")
S = json.load(open(PILOT / "summary.json"))
SC = S["scales"]
DELTA_G = [r for r in S["points_kspace"] if r["point"] == "G"][0]["delta_BdG"]
NU_N = SC["nu_N_per_site"]
NUMBERS = {}
print("loaded pilot data:", PILOT.resolve())

ModuleNotFoundError: No module named 'kwant'

## 1. Numbers quoted in V18
Quantization at the gapped point, pocket statistics, strong-disorder slopes against the class-D calibration, clean pocket channel counting, wall filtering lengths, texture winding numbers, finite-temperature conductance and KPM spectra.

In [ ]:
# G: quantization, Bott, decay length, DOS
rows = []
for U in sorted(g.U.unique()):
    dG = g[(g.point == "G") & (g.U == U)].groupby("L").lg.mean()
    sl = np.polyfit(dG.index.values[1:4], dG.values[1:4], 1)[0]
    qq = q[(q.point == "G") & (q.U == U) & (q.L == 128)]
    bb = bott[(bott.point == "G") & (bott.U == U) & (bott.L == 48)]
    rows.append(dict(U=U, Q2=f"{int((np.round(qq.Q) == 2).sum())}/{len(qq)}", min_det=qq.min_abs_det_r.min(),
                     bott_mean=bb.bott_signed.mean(), decay=-1 / sl,
                     rho0=float(dos0[(dos0.point == "G") & (dos0.U == U)].rho0.iloc[0]),
                     lng160=dG.loc[160]))
TAB_G = pd.DataFrame(rows); NUMBERS["G_table"] = TAB_G.round(4).to_dict("records")
print(TAB_G.round(3).to_string(index=False))

# pocket points: Q statistics and DOS at L = 128
rows = []
for p in ["P1", "P2", "P3", "P4"]:
    for U in sorted(g.U.unique()):
        qq = q[(q.point == p) & (q.U == U) & (q.L == 128)]
        rows.append(dict(point=p, U=U, med_det=qq.min_abs_det_r.median(),
                         fQ=float(np.mean((np.round(qq.Q) == 2) & (qq.min_abs_det_r > 0.2))),
                         g160=g[(g.point == p) & (g.U == U) & (g.L == 160)].g.mean(),
                         rho0=float(dos0[(dos0.point == p) & (dos0.U == U)].rho0.iloc[0])))
TAB_P = pd.DataFrame(rows); NUMBERS["pocket_table"] = TAB_P.round(6).to_dict("records")

# strong-disorder log slopes vs calibration (L >= 40, bootstrap)
rng = np.random.default_rng(0)


def slope(d, Lmin=40):
    d = d[d.L >= Lmin]; Ls = sorted(d.L.unique()); grs = [d[d.L == L].g.values for L in Ls]
    f = lambda ms: np.polyfit(np.log(Ls), ms, 1)[0]
    return float(f([x.mean() for x in grs])), float(np.std([f([rng.choice(x, len(x)).mean() for x in grs]) for _ in range(500)]))


SLOPES = {f"{p}|U=3": slope(g[(g.point == p) & (g.U == 3.0)]) for p in POINTS}
SLOPES.update({f"calib|{m}": slope(cal[cal.model == m]) for m in cal.model.unique()})
NUMBERS["slopes"] = SLOPES
print({k: (round(v[0], 3), round(v[1], 3)) for k, v in SLOPES.items()})

# clean ballistic channel counting
kk = np.linspace(-2.3, 2.3, 1841); KX, KY = np.meshgrid(kk, kk, indexing="ij"); dk = kk[1] - kk[0]
CH = {}
for p in ["P2", "P3", "P4"]:
    xs, xa, D = rc.bdg_fields(KX, KY, dict(LR, w=POINTS[p]))
    lab, n = ndimage.label(np.sqrt(xs ** 2 + np.abs(D) ** 2) - np.abs(xa) < 0)
    dky = sum((lab == i).any(axis=0).sum() * dk for i in range(1, n + 1))
    d = g[(g.point == p) & (g.U == 0)].set_index("L").g
    W = np.array([np.sqrt(3) * wy(L) for L in d.index])
    CH[p] = dict(sum_dky=dky, W=W, g=d.values, slope=float(np.polyfit(W, d.values, 1)[0]), pred=dky / (2 * np.pi))
NUMBERS["channels"] = {p: dict(sum_dky=v["sum_dky"], slope=v["slope"], pred=v["pred"], ratio=v["slope"] / v["pred"]) for p, v in CH.items()}
print(NUMBERS["channels"])

# walls
wm = walls.groupby(["variant", "U", "Lx"]).T.agg(["mean", "std", "count"])
NUMBERS["walls"] = walls.groupby(["variant", "U", "Lx"]).T.mean().round(5).to_dict().__repr__()


def excess_len(v, U):
    e = (wm.loc[(v, U)]["mean"] - 4).values
    Lx = wm.loc[(v, U)].index.values
    ok = e > 1e-3
    return float(-1 / np.polyfit(Lx[ok], np.log(e[ok]), 1)[0]) if ok.sum() >= 2 else np.nan


NUMBERS["wall_filter_length"] = {f"{v}|U={U}": excess_len(v, U) for v in ("normal strip (2)", "normal strip (4)", "suppressed pairing (3, 0.3)") for U in (0.0, 0.5, 1.0)}
print(NUMBERS["wall_filter_length"])

# textures: winding at L = 64
PHIS = list(np.linspace(0, 2 * np.pi, 24, endpoint=False))
texQ = []
for f in (0.5, 0.25):
    for e in (3.0, 6.0):
        for ts in range(1, 9):
            r = load("task_winding", dict(key=cyl_key(POINTS["G"], 64, leads="chains", texture="random", ell_d=e, frac_d=f, tex_seed=ts),
                                          seeds=[0], U=0.0, phis=PHIS))
            if r:
                texQ.append(dict(frac=f, ell=e, Q=r[0]["Q"], det=r[0]["min_abs_det_r"]))
texQ = pd.DataFrame(texQ)
NUMBERS["texture_Q"] = texQ.groupby(["frac", "ell"]).apply(lambda x: dict(Q2=int((np.round(x.Q) == 2).sum()), n=len(x), med_det=float(x.det.median()))).to_dict().__repr__()
print(NUMBERS["texture_Q"])

# finite temperature from cached energy-resolved transmissions
E_T = [E0] + list(0.9 * (np.arange(1, 28) / 27) ** 2)
TH = np.geomspace(2.5e-3, 0.08, 40)
KT = {}
for (p, U) in [("G", 0.5), ("P1", 0.5), ("P2", 1.0), ("P3", 1.0)]:
    for L in (64, 128):
        arr = []
        for ch in chunks(seeds_for(U, 8), 2):
            r = load("task_transport", dict(key=cyl_key(POINTS[p], L), seeds=ch, U=U, E_list=E_T))
            if r:
                arr += [x["T"] for x in r]
        if arr:
            Tm = np.mean(arr, 0)
            KT[(p, U, L)] = np.array([rc.thermal_from_T(np.array(E_T), Tm, t) for t in TH])

# KPM densities of states
E_dos = list(np.linspace(-0.4, 0.4, 81))
DOS = {}
for p, U in [("G", 0.0), ("G", 1.0), ("G", 2.0), ("P2", 0.0), ("P2", 1.0)]:
    r = load("task_kpm_dos", dict(key=torus_key(POINTS[p], 96), seed=(0 if U == 0 else 1), U=U, energies=E_dos,
                                  num_moments=3000, num_vectors=4))
    if r is not None:
        DOS[(p, U)] = np.asarray(r) / (2 * NU_N)

# KPM real-space Hall function at G vs k space
E_k = list(np.linspace(-1.0, 1.0, 41))
kkey = rc.gkey(dict(LR, w=POINTS["G"], kind="flake", Lx=60, Wy=wy(60), frame="lab"))
kp = [load("task_kpm_sxy", dict(key=kkey, U=0.0, seed=s, energies=E_k, num_moments=2000, num_vectors=3, temperature=0.002)) for s in range(8)]
kp = -np.array([x for x in kp if x is not None])
KPM_G = (kp.mean(0), kp.std(0) / np.sqrt(len(kp)))
E_conv = np.linspace(-30, 30, 30001)
KTH_G = rc.kspace_thermal(dict(LR, w=POINTS["G"]), Nk=401, thetas=[0.01], E_grid=E_conv)
NUMBERS["kpm_G_s0"] = [float(KPM_G[0][20]), float(KPM_G[1][20])]
print("KPM s(0) at G:", NUMBERS["kpm_G_s0"])

## 2. Main Fig. 4: disorder test of the transport window

In [ ]:
U_RAMP = {0.5: BLUE_RAMP[0], 1.0: BLUE_RAMP[1], 1.5: BLUE_RAMP[3], 2.0: BLUE_RAMP[4]}
fig, ax = plt.subplots(2, 2, figsize=(7.2, 5.6))
a = ax[0, 0]
for U, c in U_RAMP.items():
    d = g[(g.point == "G") & (g.U == U)].groupby("L").lg.mean()
    a.semilogy(d.index, np.exp(d.values), "o--", color=c, mfc="white", ms=3.6, lw=1.3, label=rf"G, $U={U:g}$")
for p in ["P1", "P2", "P3", "P4"]:
    d = g[(g.point == p) & (g.U == 1.0)].groupby("L").lg.mean()
    a.semilogy(d.index, np.exp(d.values), MK[p] + "-", color=C_P[p], ms=3.8, label=rf"{p}, $U=1$")
a.set_xlabel(r"sample size $L$"); a.set_ylabel(r"typical $g=e^{\langle\ln\mathcal{T}\rangle}$")
a.set_ylim(1e-17, 1e2); a.legend(ncol=2, loc="lower left", fontsize=7.2, handlelength=1.8, columnspacing=0.8)
a.text(158, 0.25, "pockets: heat-conducting bulk", ha="right", va="top", fontsize=7.8, color=C_P["P3"])
a.text(158, 2e-4, "gapped: evanescent bulk", ha="right", va="top", fontsize=7.8, color=C_G)
panel(a, "a")

a = ax[0, 1]
Us = sorted(g.U.unique())[1:]
dG = [q[(q.point == "G") & (q.U == U) & (q.L == 128)].min_abs_det_r.median() for U in Us]
a.semilogy(Us, dG, MK["G"] + "-", color=C_G, label="G (gapped)", zorder=3)
for p in ["P1", "P2", "P3", "P4"]:
    a.semilogy(Us, [q[(q.point == p) & (q.U == U) & (q.L == 128)].min_abs_det_r.median() for U in Us],
               MK[p] + "-", color=C_P[p], label=p)
a.axvspan(0.15, 2.1, color=C_G, alpha=0.06, lw=0)
a.text(1.1, 2.2, r"$Q=2$ in every sample", color=C_G, ha="center", fontsize=7.8)
a.set_ylim(1e-13, 20); a.set_xlabel(r"disorder $U$"); a.set_ylabel(r"median $\min_\phi|\det r(\phi)|$")
a.legend(loc="lower right", ncol=2, fontsize=7.2); panel(a, "b")

a = ax[1, 0]
Ed = np.array(E_dos) / DELTA_G
for (p, U), c, ls in [(("G", 0.0), BLUE_RAMP[0], "-"), (("G", 1.0), BLUE_RAMP[2], "-"), (("G", 2.0), BLUE_RAMP[4], "-"),
                      (("P2", 0.0), C_P["P2"], "--"), (("P2", 1.0), C_P["P3"], "--")]:
    if (p, U) in DOS:
        a.plot(Ed, DOS[(p, U)], ls, color=c, label=rf"{p}, $U={U:g}$")
a.axvline(0, color=INK2, lw=0.6)
a.set_xlim(-2.4, 2.4); a.set_ylim(0, 0.5)
a.set_xlabel(r"$E/\delta_{\rm BdG}^{\rm G}$"); a.set_ylabel(r"$\rho(E)/2\nu_N$")
a.annotate("G, $U=2$:\n" + r"$\rho(0)\neq0$, still $Q=2$", xy=(0.0, 0.028), xytext=(-2.3, 0.035), fontsize=7.4,
           color=BLUE_RAMP[4], arrowprops=dict(arrowstyle="->", color=BLUE_RAMP[4], lw=0.8))
a.legend(loc="upper center", fontsize=6.9, ncol=2, columnspacing=0.8, handlelength=1.6); panel(a, "c")

a = ax[1, 1]
for (p, U), c, m in [(("G", 0.5), C_G, "o"), (("P1", 0.5), C_P["P1"], "s"), (("P2", 1.0), C_P["P2"], "D"), (("P3", 1.0), C_P["P3"], "^")]:
    K = KT.get((p, U, 128))
    if K is None:
        continue
    ok = K > 1e-4
    a.loglog(TH[ok] / DELTA_G, K[ok], "-", color=c, label=rf"{p}, $U={U:g}$")
a.set_xlabel(r"$k_BT/\delta_{\rm BdG}^{\rm G}$"); a.set_ylabel(r"$K/(q_MT)$, $L=128$")
a.set_ylim(1e-3, 30); a.legend(loc="lower right", fontsize=7.2)
a.xaxis.set_major_locator(mpl.ticker.FixedLocator([0.02, 0.05, 0.1, 0.2, 0.5]))
a.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%g"))
a.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
a.text(0.04, 0.62, "pockets: finite residual\nas $T\\to0$", transform=a.transAxes, fontsize=7.8, color=C_P["P3"], va="top")
a.text(0.04, 0.22, "gapped:\nactivated", transform=a.transAxes, fontsize=7.8, color=C_G)
panel(a, "d")
fig.tight_layout(w_pad=1.6, h_pad=1.2)
fig.savefig(FIGDIR / "fig_kwant_disorder_window.pdf"); fig.savefig(FIGDIR / "fig_kwant_disorder_window.png", dpi=170)
plt.close(fig)

## 3. Main Fig. 5: written walls and domain networks

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(7.2, 5.6))
a = ax[0, 0]
VAR = [("sharp wall", C_G, "o", "sharp"), ("suppressed pairing (3, 0.3)", C_AQ, "s", "suppressed pairing"),
       ("normal strip (2)", C_P["P2"], "D", "metallic core, width 2"), ("normal strip (4)", C_P["P4"], "^", "metallic core, width 4"),
       ("erased (uniform)", "#8a8985", "v", "erased")]
for v, c, m, lab in VAR:
    for U, ls, fc in [(0.0, "-", c), (1.0, "--", "white")]:
        d = wm.loc[(v, U)]["mean"]
        a.plot(d.index, d.values, m + ls, color=c, mfc=fc, ms=3.8, lw=1.3, label=lab if U == 0 else None)
a.axhline(4, color="k", lw=0.7, ls=":")
a.text(165, 4.22, r"$2|C|=4$", fontsize=8, ha="center")
a.set_xlabel(r"wall length $L_x$"); a.set_ylabel(r"$K_{\rm DW}/(q_MT)$")
a.set_ylim(-0.2, 6.6); a.legend(loc="center right", bbox_to_anchor=(1.0, 0.36), fontsize=7.0)
a.text(0.03, 0.08, r"solid: clean; dashed: $U=1$", transform=a.transAxes, fontsize=7.4, color=INK2)
panel(a, "a")

a = ax[0, 1]
mkey = rc.gkey(dict(LR, w=POINTS["G"], kind="cylinder", Lx=64, Wy=24, frame="lab", buffer=BUFFER, lead_type="metal",
                    t_lead=1.0, texture="wall2"))
fs_m = rc.cached_system(mkey)
psi = kwant.wave_function(fs_m, energy=0.02, params=rc._params(0, 0))(0)
Jop = kwant.operator.Current(fs_m)
cur = sum(Jop(ps, params=rc._params(0, 0)) for ps in psi)
cmap = mpl.colors.LinearSegmentedColormap.from_list("blues", ["#fcfcfb", "#86b6ef", "#2a78d6", "#0d366b"])
kwant.plotter.current(fs_m, cur, ax=a, colorbar=False, cmap=cmap)
Hm = np.sqrt(3) * 24
for yw in (Hm / 4, 3 * Hm / 4):
    a.axhline(yw, color=C_P["P3"], ls=":", lw=0.9)
a.text(62, 3 * Hm / 4 + 1.5, "wall", color=C_P["P3"], fontsize=7.6, ha="right")
a.text(62, Hm / 4 + 1.5, "wall", color=C_P["P3"], fontsize=7.6, ha="right")
a.text(33, Hm / 2, r"$C=-2$", ha="center", fontsize=8, color=INK2)
a.text(33, 4, r"$C=+2$", ha="center", fontsize=8, color=INK2)
a.text(33, 36.5, r"$C=+2$", ha="center", fontsize=8, color=INK2)
a.set_xlabel("$x$"); a.set_ylabel("$y$"); panel(a, "b")

a = ax[1, 0]
prof = rc.cut_current_profile(mkey, 0.02)
bins = np.linspace(0, Hm, 29)
h, _ = np.histogram(prof[:, 0], bins=bins, weights=prof[:, 1])
a.bar(0.5 * (bins[1:] + bins[:-1]), h, width=0.9 * (bins[1] - bins[0]), color=C_G)
for yw in (Hm / 4, 3 * Hm / 4):
    a.axvline(yw, color=C_P["P3"], ls=":", lw=0.9)
a.set_xlabel(r"$y$ (cut at $x=L_x/2$)"); a.set_ylabel("quasiparticle flux")
a.text(0.04, 0.9, rf"total $=\,{prof[:, 1].sum():.3f}$", transform=a.transAxes, fontsize=8.2)
NUMBERS["cut_total"] = float(prof[:, 1].sum())
panel(a, "c")

a = ax[1, 1]
for f, c in [(0.5, C_AQ), (0.25, C_G)]:
    for e, ls, fc in [(3.0, "-", c), (6.0, "--", "white")]:
        d = tex[(tex.frac == f) & (tex.ell == e)].groupby("L").g.mean()
        a.semilogy(d.index, d.values, "o" + ls, color=c, mfc=fc, ms=3.8, lw=1.3)
a.set_xlabel(r"sample size $L$"); a.set_ylabel(r"$\langle K\rangle/(q_MT)$")
a.set_ylim(8e-3, 2e3)
a.text(126, 4.8, "50% reversed: percolating walls,\n$Q$ not quantized", ha="right", va="bottom", fontsize=7.2, color=C_AQ)
a.text(26, 0.012, "25% reversed: islands,\n$Q=2$ in 16/16", ha="left", va="bottom", fontsize=7.2, color=C_G)
a.text(0.03, 0.95, "solid: $\\ell_d=3$\ndashed: $\\ell_d=6$", transform=a.transAxes, fontsize=7.2, color=INK2, va="top")
for (f, x0) in [(0.5, 0.36), (0.25, 0.70)]:
    ins = a.inset_axes([x0, 0.70, 0.27, 0.21])
    m = rc.domain_mask(1, 2 * 64, 2 * wy(64), 3.0, f, True)
    ins.imshow(m.T, origin="lower", cmap=mpl.colors.ListedColormap(["#e6e5e1", C_AQ if f == 0.5 else C_G]),
               aspect="auto", interpolation="nearest")
    ins.set_xticks([]); ins.set_yticks([])
    ins.set_title("percolating" if f == 0.5 else "islands", fontsize=7, pad=1.5)
panel(a, "d")
fig.tight_layout(w_pad=1.6, h_pad=1.2)
fig.savefig(FIGDIR / "fig_kwant_writable_walls.pdf"); fig.savefig(FIGDIR / "fig_kwant_writable_walls.png", dpi=170)
plt.close(fig)
print("main figures written")

## 4. Appendix figure: lattice-resolved model, pocket channel counting, real-space Hall function

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(7.2, 5.6))
a = ax[0, 0]
ws = np.linspace(0, 4.5, 46)
dl = [rc.kspace_summary(dict(LR, w=float(w)), 121, 301)["delta_BdG"] for w in ws]
a.plot(ws, dl, "-", color=INK2, lw=1.3)
a.axhline(0, color="k", lw=0.6)
for p, w in POINTS.items():
    d = [r for r in S["points_kspace"] if r["point"] == p][0]["delta_BdG"]
    a.plot(w, d, MK[p], color=C_G if p == "G" else C_P[p], ms=6, zorder=3)
    a.annotate(p, (w, d), xytext=(5, 3), textcoords="offset points", fontsize=8)
a.axvline(SC["w_c"], color=INK2, ls=":", lw=0.9)
a.text(SC["w_c"] + 0.05, -0.75, rf"$w_c={SC['w_c']:.2f}$", fontsize=8)
a.set_xlabel("trigonal warping $w$"); a.set_ylabel(r"$\delta_{\rm BdG}$")
a.text(0.97, 0.95, r"$C_{\rm band}=2$ at every $w$", transform=a.transAxes, ha="right", va="top", fontsize=8)
panel(a, "a")

a = ax[0, 1]
kc = np.linspace(-2.2, 2.2, 701); CX, CY = np.meshgrid(kc, kc, indexing="ij")
xs, xa, D = rc.bdg_fields(CX, CY, dict(LR, w=POINTS["P2"]))
dmap = np.sqrt(xs ** 2 + np.abs(D) ** 2) - np.abs(xa)
div = mpl.colors.LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
im = a.pcolormesh(CX, CY, dmap, cmap=div, vmin=-0.5, vmax=0.5, shading="auto", rasterized=True)
a.contour(CX, CY, dmap, levels=[0], colors="k", linewidths=1.1)
a.contour(CX, CY, xs, levels=[0], colors=INK2, linewidths=0.7, linestyles="--")
a.set_aspect("equal"); a.set_xlabel("$k_x$"); a.set_ylabel("$k_y$")
cb = fig.colorbar(im, ax=a, fraction=0.046, pad=0.03); cb.set_label(r"$\eta-|\xi_a|$", fontsize=8)
a.text(0.03, 0.95, "P2", transform=a.transAxes, fontsize=8.5, va="top")
panel(a, "b")

a = ax[1, 0]
for p in ["P2", "P3", "P4"]:
    v = CH[p]
    a.plot(v["W"], v["g"], MK[p], color=C_P[p], label=f"{p} (clean)")
    Wg = np.linspace(0, v["W"].max() * 1.05, 10)
    a.plot(Wg, Wg * v["pred"], "--", color=C_P[p], lw=1.0)
a.set_xlabel("sample width $W$"); a.set_ylabel(r"$K/(q_MT)$, $U=0$")
a.text(0.03, 0.95, r"dashed: $W\sum\Delta k_y/2\pi$ (pocket channels)", transform=a.transAxes, va="top", fontsize=7.6)
a.legend(loc="center left", bbox_to_anchor=(0.0, 0.62)); panel(a, "c")

a = ax[1, 1]
sE = KTH_G["s_E"]; sig = np.pi * 26.7 / 2000
ker = np.exp(-0.5 * ((E_conv[:, None] - np.array(E_k)[None, :]) / sig) ** 2)
sb = (ker * sE[:, None]).sum(0) / ker.sum(0)
a.plot(np.array(E_k) / DELTA_G, sb, "-", color=INK2, label="k space (broadened)")
a.errorbar(np.array(E_k) / DELTA_G, KPM_G[0], yerr=KPM_G[1], fmt="o", ms=3.2, color=C_G, label="real-space Kubo-Bastin (Kwant KPM)")
a.axhline(2, color="k", lw=0.6, ls=":")
a.set_xlabel(r"$E/\delta_{\rm BdG}^{\rm G}$"); a.set_ylabel("$s(E)$")
a.set_xlim(-3.6, 3.6); a.set_ylim(0.6, 3.3); a.legend(loc="lower center", fontsize=7.2)
a.text(0.03, 0.95, rf"G: $s(0)={KPM_G[0][20]:.3f}\pm{KPM_G[1][20]:.3f}$", transform=a.transAxes, va="top", fontsize=8)
panel(a, "d")
fig.tight_layout(w_pad=1.6, h_pad=1.2)
fig.savefig(FIGDIR / "fig_kwant_model_bridge.pdf"); fig.savefig(FIGDIR / "fig_kwant_model_bridge.png", dpi=170)
plt.close(fig)

## 5. Appendix figure: full disorder scaling and class-D calibration

In [ ]:
URAMP7 = ["#b7d3f6", "#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
Uall = sorted(g.U.unique())
fig, ax = plt.subplots(2, 3, figsize=(7.2, 4.9), sharex=True)
for a, p in zip(ax.flat[:5], ["G", "P1", "P2", "P3", "P4"]):
    for c, U in zip(URAMP7, Uall):
        d = g[(g.point == p) & (g.U == U)].groupby("L").g.agg(["mean", "std", "count"])
        if p == "G":
            dd = g[(g.point == p) & (g.U == U)].groupby("L").lg.mean()
            a.semilogy(dd.index, np.exp(dd.values), "o-", color=c, ms=2.8, lw=1.1, label=f"$U={U:g}$")
        else:
            a.errorbar(d.index, d["mean"], yerr=d["std"] / np.sqrt(d["count"]), fmt="o-", color=c, ms=2.8, lw=1.1,
                       capsize=1.5, label=f"$U={U:g}$")
    a.set_xscale("log")
    a.xaxis.set_major_locator(mpl.ticker.FixedLocator([20, 40, 80, 160]))
    a.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%g"))
    a.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    a.set_title(rf"{p}: $w={POINTS[p]:g}$", fontsize=9)
    if p != "G":
        a.set_yscale("log"); a.set_ylim(0.2, 50)
    else:
        a.set_ylim(1e-17, 1e2)
ax[0, 0].set_ylabel(r"typical $g$"); ax[1, 0].set_ylabel(r"$\langle g\rangle$")
ax[0, 0].legend(fontsize=5.8, loc="lower left", ncol=2, columnspacing=0.6, handlelength=1.2)
a = ax[1, 2]
for m, c, mk in [("metal: mu=4t, U=3t, D=t", C_AQ, "o"), ("metal: mu=4t, U=3t, D=t/2", "#008300", "s")]:
    d = cal[cal.model == m].groupby("L").g.agg(["mean", "std", "count"])
    b, e = SLOPES[f"calib|{m}"]
    a.errorbar(d.index, d["mean"], yerr=d["std"] / np.sqrt(d["count"]), fmt=mk + "-", color=c, ms=3, capsize=1.5,
               label=rf"$p+ip$ metal, $b={b:.2f}\pm{e:.2f}$")
for p in ["P2", "P3"]:
    d = g[(g.point == p) & (g.U == 3.0)].groupby("L").g.mean()
    b, e = SLOPES[f"{p}|U=3"]
    a.plot(d.index, d.values, MK[p] + "--", color=C_P[p], ms=3, label=rf"{p}, $U=3$, $b={b:.2f}\pm{e:.2f}$")
a.set_xscale("log"); a.set_title(r"class-D calibration", fontsize=9)
a.xaxis.set_major_locator(mpl.ticker.FixedLocator([20, 40, 80, 160]))
a.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%g"))
a.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
a.set_ylim(1.4, 3.9); a.legend(fontsize=5.8, loc="upper left")
for a in ax[1]:
    a.set_xlabel("$L$")
fig.tight_layout(w_pad=0.8, h_pad=0.8)
fig.savefig(FIGDIR / "fig_kwant_scaling_appendix.pdf"); fig.savefig(FIGDIR / "fig_kwant_scaling_appendix.png", dpi=170)
plt.close(fig)
(FIGDIR / "kwant_v18_numbers.json").write_text(json.dumps(NUMBERS, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)))
print("all figures written to", FIGDIR.resolve())